<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/15_information_extraction_with_llms_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Information Extraction with an LLM

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Classification gives each text one label. Many research questions need more: **who** is quoted in a
news story, **which** party they belong to, **what** they said. Information extraction turns each text
into one or more **records** with fields you define, ready to analyze like any other dataset.

In this notebook an LLM reads BBC political news from 2004–2005 and extracts every direct quotation:
who said it, their role, their party, and the exact words. That's enough to start on a classic
question in journalism research: **who gets a voice in the news?**

> ⚠️ **Turn on the GPU first:** *Runtime → Change runtime type → T4 GPU.*

In this notebook you will:

- design an extraction schema: a codebook for records instead of labels
- see how often free-form JSON goes wrong, and fix the format with structured outputs in Ollama
- check every extracted quote against the article, to catch altered or invented quotes
- measure recall against a simple pattern, and check the speakers by hand
- turn the records into a dataset and ask who gets quoted

**Contents**

1. Classification versus extraction
2. The data
3. A local model with Ollama
4. A first try: just ask for JSON
5. Structured outputs
6. Running it, with a cache
7. Is it really in the text?
8. Did it find everything?
9. Who gets a voice?
10. Wrap-up

There are 5 exercises along the way, marked ✏️. The solutions notebook has worked answers, but try them yourself first.

## 0. Setup

In [ ]:
!pip install -q ollama pydantic

import json
import os
import re
import subprocess
import time
from difflib import SequenceMatcher
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from huggingface_hub import hf_hub_download
from pydantic import BaseModel, Field, ValidationError

SEED = 42
pd.set_option("display.max_colwidth", 140)

---
## 1. Classification versus extraction

| | Classification | Extraction |
|---|---|---|
| Output per text | one label | a list of records, each with several fields |
| Example | the policy area of a bill | every quote: speaker, role, party, words |
| The codebook | label definitions | a **schema**: fields, allowed values, rules |
| Validation | accuracy per label | per record (found? invented?) and per field (right speaker?) |

Other examples from social science: actors and the claims they make, protest events (who, where,
when, how many), relations ("who criticizes whom"), or the policies a party promises. The recipe is
the same as in this notebook; only the schema changes.

---
## 2. The data

About 2,200 BBC News articles from 2004 and 2005 in five sections, first collected by Greene &
Cunningham (2006). We use the politics section: the run-up to the 2005 UK general election. In this copy of the
data, the title is followed by `..` and the paragraphs are joined with `..`, so we restore them first.

In [ ]:
path = hf_hub_download(repo_id="gopalkalpande/bbc-news-summary", repo_type="dataset",
                       revision="refs/convert/parquet", filename="default/train/0000.parquet")
bbc = pd.read_parquet(path)
print(bbc["File_path"].value_counts())


def split_article(raw):
    title, body = raw.split("..", 1)
    paragraphs = re.split(r'(?<=[.!?"\'])\.\.(?!\.)', body)
    return title.strip(), "\n\n".join(p.strip() for p in paragraphs if p.strip())


politics = bbc[bbc["File_path"] == "politics"].copy()
politics[["title", "text"]] = pd.DataFrame([split_article(a) for a in politics["Articles"]], index=politics.index)
politics = politics.drop_duplicates("text").reset_index(drop=True)
politics["article_id"] = politics.index
texts = politics.set_index("article_id")["text"]
print(f"\n{len(politics)} politics articles, median {politics.text.str.split().str.len().median():.0f} words\n")
print(politics.title[0], "\n\n" + politics.text[0][:1000])

Quotation marks are easy to find with a pattern. Passages of five words or more in quotation marks
are mostly speech; shorter ones are often titles, slogans or scare quotes ("so-called reform"). The
pattern can't tell **who** is speaking, which is what we need the model for, but it gives us a cheap
check on the model's recall later.

In [ ]:
def quoted_passages(text, min_words=5):
    return [q.strip() for q in re.findall(r'"([^"]+)"', text) if len(q.split()) >= min_words]


politics["n_quoted"] = politics["text"].map(lambda t: len(quoted_passages(t)))
print(politics["n_quoted"].describe().round(1))
quoted_passages(politics.text[0])[:5]

---
## 3. A local model with Ollama

### Starting Ollama

The model runs locally through **Ollama**, as in the optional part of the Day 4 notebook. Colab is
Linux, so we install it here; on your own laptop you'd install the app from
[ollama.com](https://ollama.com) instead. The first cell takes about a minute, the download of the
model a minute or two more.

In [ ]:
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y pciutils zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):                                       # wait until Ollama answers
    try:
        print(requests.get("http://localhost:11434", timeout=1).text)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

We use **Qwen2.5 7B**, quantized to 4 bits by Ollama: big enough to follow a schema and copy quotes
reliably, small enough for Colab's free GPU. To go faster, try `qwen2.5:3b` (Exercise 5).

In [ ]:
MODEL = "qwen2.5:7b"
!ollama pull {MODEL}

In [ ]:
import ollama


def ask(messages, schema=None, max_tokens=1500):
    return ollama.chat(model=MODEL, messages=messages, format=schema,
                       options={"temperature": 0, "num_ctx": 4096, "num_predict": max_tokens})


print(ask([{"role": "user", "content": "In one sentence: what is a direct quotation?"}]).message.content)

`num_ctx` sets the context window. Ollama's default is small; an article plus instructions plus the
answer needs about 2,000 tokens, so we make room for 4,096.

---
## 4. A first try: just ask for JSON

The instructions are our codebook: what counts as a quote, and what to record about it.

In [ ]:
INSTRUCTIONS = """You extract direct quotations from news articles.
A direct quotation is text in quotation marks that the article presents as the words of a person.
For every direct quotation, give:
- speaker: the person's name as written in the article, or a short description if no name is given
- role: politician, government official, expert or academic, business, campaign group or union,
  member of the public, or other
- party: Labour, Conservative, Liberal Democrat, other party, or no party
- quote: the quoted words, copied exactly
Skip words in quotation marks that nobody is presented as saying, such as titles, slogans and scare quotes."""

FIRST_TRY = INSTRUCTIONS + "\nReply with a JSON list of objects with the keys speaker, role, party and quote."
ROLES = ["politician", "government official", "expert or academic", "business", "campaign group or union",
         "member of the public", "other"]
PARTIES = ["Labour", "Conservative", "Liberal Democrat", "other party", "no party"]


def check_answer(answer):
    """Is this answer a list of records with the right keys, and only allowed values?"""
    try:
        records = json.loads(re.sub(r"^```(?:json)?|```$", "", answer.strip(), flags=re.M).strip())
    except json.JSONDecodeError:
        return "not valid JSON"
    if not isinstance(records, list) or not all(isinstance(r, dict) for r in records):
        return "valid JSON, wrong shape"
    if not all({"speaker", "role", "party", "quote"} <= set(r) for r in records):
        return "missing keys"
    if not all(r["role"] in ROLES and r["party"] in PARTIES for r in records):
        return "values outside the codebook"
    return "usable"


trial = politics.sample(8, random_state=SEED)
raw_answers = [ask([{"role": "system", "content": FIRST_TRY}, {"role": "user", "content": t}]).message.content
               for t in trial.text]
pd.Series([check_answer(a) for a in raw_answers]).value_counts()

> **✏️ Exercise 1**
>
> Print the answers that aren't `usable`, or the first two if they all are. What went wrong, and how
> much cleaning code would it take to fix answers like these across a few thousand articles?

In [ ]:
# Your code here

---
## 5. Structured outputs

Ollama can force the answer to follow a **JSON schema**: while the model writes, any token that would
break the schema is simply not allowed. We write the schema as a **Pydantic** class, so the codebook
becomes code. `Literal` fields accept only the listed values. The same class later checks every answer.

The schema fixes the **format**, not the **content**: the model can still pick the wrong speaker or
miss a quote, so the instructions stay in the prompt.

In [ ]:
Role = Literal["politician", "government official", "expert or academic", "business",
               "campaign group or union", "member of the public", "other"]
Party = Literal["Labour", "Conservative", "Liberal Democrat", "other party", "no party"]


class Quote(BaseModel):
    speaker: str = Field(description="name of the person quoted, as written in the article")
    role: Role
    party: Party
    quote: str = Field(description="the quoted words, copied exactly from the article")


class ArticleQuotes(BaseModel):
    quotes: list[Quote]


print(json.dumps(ArticleQuotes.model_json_schema(), indent=1)[:700], "...")

In [ ]:
def extract(text, schema=ArticleQuotes, instructions=INSTRUCTIONS):
    reply = ask([{"role": "system", "content": instructions}, {"role": "user", "content": text}],
                schema=schema.model_json_schema())
    return schema.model_validate_json(reply.message.content), reply


result, reply = extract(trial.text.iloc[0])
print(f"{reply.prompt_eval_count} tokens in, {reply.eval_count} out, {reply.total_duration / 1e9:.1f} s\n")
pd.DataFrame([q.model_dump() for q in result.quotes])

---
## 6. Running it, with a cache

Forty articles take a few minutes. Every result is written to a file as soon as it's done, so if Colab
disconnects you rerun the cell and it carries on where it stopped (Bonus 13 has more on this). An
answer that fails validation is recorded as an error rather than stopping the loop.

In [ ]:
SAMPLE_SIZE = 40
sample = politics.sample(SAMPLE_SIZE, random_state=SEED)
CACHE = "quotes_cache.jsonl"

done = {}
if os.path.exists(CACHE):
    with open(CACHE) as f:
        done = {r["article_id"]: r for r in map(json.loads, f)}

with open(CACHE, "a") as f:
    for aid, text in zip(sample.article_id, sample.text):
        if aid in done:
            continue
        start = time.time()
        try:
            result, _ = extract(text)
            record = {"article_id": int(aid), "quotes": [q.model_dump() for q in result.quotes], "error": None}
        except ValidationError as e:
            record = {"article_id": int(aid), "quotes": [], "error": str(e)[:300]}
        record["seconds"] = round(time.time() - start, 1)
        done[aid] = record
        f.write(json.dumps(record) + "\n")

quotes = pd.DataFrame([{"article_id": aid, **q} for aid, rec in done.items() for q in rec["quotes"]])
print(f"{len(quotes)} quotes from {len(done)} articles, {sum(r['error'] is not None for r in done.values())} errors, "
      f"{np.mean([r['seconds'] for r in done.values()]):.1f} s per article")
quotes.head(10)

---
## 7. Is it really in the text?

An LLM can "tidy up" a quote, merge two quotes, or produce words nobody said. Because we asked for
exact words, we can check every quote automatically:

- **exact**: the quote appears word for word in the article (ignoring case and spacing);
- **partly**: a long stretch of it does, at least 60% of its characters;
- **not in text**: neither.

We also check whether the speaker's last name appears in the article at all.

In [ ]:
def normalize(s):
    s = s.replace("\u201c", '"').replace("\u201d", '"').replace("\u2019", "'").replace("\u2018", "'")
    return re.sub(r"\s+", " ", s).strip().strip('"').strip().lower()


def grounding(article_id, quote):
    text, q = normalize(texts[article_id]), normalize(quote)
    if q and q in text:
        return "exact"
    m = SequenceMatcher(None, text, q, autojunk=False).find_longest_match(0, len(text), 0, len(q))
    return "partly" if q and m.size >= 0.6 * len(q) else "not in text"


quotes["grounding"] = [grounding(a, q) for a, q in zip(quotes.article_id, quotes.quote)]
quotes["speaker_in_text"] = [str(s).split()[-1].lower() in texts[a].lower() if str(s).split() else False
                             for a, s in zip(quotes.article_id, quotes.speaker)]
print(quotes["grounding"].value_counts(normalize=True).round(2), "\n")
print(f"Speaker's last name appears in the article: {quotes.speaker_in_text.mean():.0%}")

> **✏️ Exercise 2**
>
> Print the quotes that are not `exact`, each next to the closest passage in the article (use
> `SequenceMatcher(...).find_longest_match` as in `grounding`). What did the model change? What would
> you do with these quotes in a real project?

In [ ]:
# Your code here

---
## 8. Did it find everything?

Grounding says whether what the model found is real; it says nothing about what it missed. For that we
use the quoted passages from Section 2: a passage counts as **found** if it appears inside one of the
model's quotes, or overlaps one for most of its length.

This is a rough standard: some quoted passages aren't speech at all, so 100% is not the target. The
misses are what's worth reading.

In [ ]:
def found(passage, extracted):
    p = normalize(passage)
    for q in map(normalize, extracted):
        m = SequenceMatcher(None, q, p, autojunk=False).find_longest_match(0, len(q), 0, len(p))
        if p in q or m.size >= 0.8 * len(p):
            return True
    return False


recall = pd.DataFrame([{"article_id": aid, "passage": p, "found": found(p, [q["quote"] for q in rec["quotes"]])}
                       for aid, rec in done.items() for p in quoted_passages(texts[aid])])
print(f"The model recovered {recall.found.mean():.0%} of {len(recall)} quoted passages of five words or more")
recall[~recall.found].head(8)

> **✏️ Exercise 3**
>
> Check the **speakers** by hand. Draw 10 quotes, print each with the 300 characters before it in the
> article, and decide whether the model named the right person. Store your verdicts in a dictionary
> (`{row index: True/False}`) and compute the accuracy. Where do the errors come from?

In [ ]:
# Your code here

---
## 9. Who gets a voice?

Now the records are a dataset. We drop quotes that weren't found in the text, and look at who is
quoted: by role, by party, and the most quoted people.

In [ ]:
usable = quotes[quotes.grounding != "not in text"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
usable["role"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="#34B233", title="Quotes by role")
(usable[usable.role == "politician"]["party"].value_counts().sort_values()
     .plot(kind="barh", ax=axes[1], color="#34B233", title="Politicians' quotes by party"))
plt.tight_layout()
plt.show()

usable["speaker"].value_counts().head(10)

A real analysis needs all the politics articles and validated speaker codes, but the pattern is a familiar one
from research on news sources: the governing party and its ministers tend to dominate, and most
voices belong to politicians and officials rather than experts or citizens. Watch out for **spelling
variants** of the same person ("Mr Brown", "Gordon Brown", "the chancellor"): counting speakers
properly means merging them first.

> **✏️ Exercise 4**
>
> Add a field to the schema: the **topic** of each quote, with the values `economy and taxes`,
> `public services`, `crime and security`, `Iraq and foreign policy`, `Europe`, `the election itself` and
> `other`. Write a new schema class, add one line about the topic to the instructions, run it on 10
> articles from `sample`, and cross-tabulate party by topic. What do the parties talk about?

In [ ]:
# Your code here

> **✏️ Exercise 5**
>
> Is a smaller model good enough? Pull `qwen2.5:3b`, run it on the first 10 articles of `sample`, and
> compare it with the 7B results on the same articles: number of quotes, share of exact quotes, recall,
> and seconds per article. Remember to set `MODEL` back afterwards.

In [ ]:
# Your code here

---
## 10. Wrap-up

- **The schema is your codebook.** Define every field, its allowed values and the decision rules,
  and pilot it on a few texts before the full run.
- **Structured outputs fix the format, not the content.** Always validate what's inside.
- **Ask for exact words, then check them.** Grounding checks are cheap and catch altered or invented
  text automatically; run them on every record, not on a sample.
- **Validate the fields** against hand coding of a random sample (Bonus 5), and report how you
  handled records that failed the checks.
- **Cache and resume**, and report the model, the schema, the instructions and the sample.

In [ ]:
server.terminate()    # stop Ollama when you're done

### Further reading

- Greene, D., & Cunningham, P. (2006). Practical solutions to the problem of diagonal dominance in
  kernel document clustering. *Proceedings of the 23rd International Conference on Machine Learning*.
  (The BBC News dataset.)
- Vaucher, T., Spitz, A., Catasta, M., & West, R. (2021). Quotebank: A corpus of quotations from a
  decade of news. *Proceedings of the 14th ACM International Conference on Web Search and Data Mining*.
- Ollama (2024). Structured outputs. [ollama.com/blog/structured-outputs](https://ollama.com/blog/structured-outputs)
- Pydantic documentation: [docs.pydantic.dev](https://docs.pydantic.dev)